In [ ]:
import pandas as pd
import sqlite3
import os
import re
import unicodedata


def normalize_column(col):

    col = col.replace("\xa0", " ")
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("ascii")
    col = col.lower()
    col = re.sub(r"[^\w\s]", "", col)
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"_+", "_", col)

    return col.strip("_")


directory = "files"

file_names = [
    "Composição do Stack Tecnológico de Dados no Cepel_Parte 0 (Identificação)(1-21).xlsx",
    "Composição do Stack Tecnológico de Dados no Cepel_Parte 1 (Geral)(1-19).xlsx",
    "Composição do Stack Tecnológico de Dados no Cepel_Parte 2 (Avançada)(1-19).xlsx",
    "Composição do Stack Tecnológico de Dados no Cepel_Parte 3 (Interface DevOps)(1-18).xlsx"
]

db_path = "cepel_data.db"
conn = sqlite3.connect(db_path)

for file_name in file_names:

    file_path = os.path.join(directory, file_name)

    df = pd.read_excel(file_path)

    # normalize columns
    df.columns = [normalize_column(c) for c in df.columns]

    table_name = (
        file_name
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("-", "_")
        .split(".")[0]
        .lower()
    )

    df.to_sql(table_name, conn, if_exists="replace", index=False)

In [ ]:
import pandas as pd
import sqlite3
import os
import re
import unicodedata

def normalize_column(col):

    col = col.replace("\xa0", " ")
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("ascii")
    col = col.lower()
    col = re.sub(r"[^\w\s]", "", col)
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"_+", "_", col)

    return col.strip("_").upper()



# Define the directory where the files are located
directory = "files"

# List of file names
file_names = [
    "Composição do Stack Tecnológico de Dados no Cepel_Parte 0 (Identificação)(1-21).xlsx",
    "Composição do Stack Tecnológico de Dados no Cepel_Parte 1 (Geral)(1-19).xlsx",
    "Composição do Stack Tecnológico de Dados no Cepel_Parte 2 (Avançada)(1-19).xlsx",
    "Composição do Stack Tecnológico de Dados no Cepel_Parte 3 (Interface DevOps)(1-18).xlsx"
]

# Create or connect to the SQLite database
db_path = "cepel_data.db"
conn = sqlite3.connect(db_path)
print(f"Connected to SQLite database at {db_path}")

# Read each Excel file and load it into the SQLite database as a separate table
for file_name in file_names:
    file_path = os.path.join(directory, file_name)
    try:
        # Read the Excel file into a pandas DataFrame
        df = pd.read_excel(file_path)
        df.columns = [normalize_column(c) for c in df.columns]
        # Create a table name from the file name (remove special characters and spaces)
        table_name = file_name.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_").split(".")[0]
        
        # Write the DataFrame to the SQLite database as a table
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"Successfully loaded {file_name} into table {table_name}")
    except FileNotFoundError:
        print(f"File {file_name} not found at {file_path}")
    except Exception as e:
        print(f"Error processing {file_name}: {str(e)}")

# Check all tables in the database to confirm their names
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("\nTables in the database:")
for table in tables:
    print(table[0])

Connected to SQLite database at cepel_data.db
Successfully loaded Composição do Stack Tecnológico de Dados no Cepel_Parte 0 (Identificação)(1-21).xlsx into table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação1_21
Successfully loaded Composição do Stack Tecnológico de Dados no Cepel_Parte 1 (Geral)(1-19).xlsx into table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_1_Geral1_19
Successfully loaded Composição do Stack Tecnológico de Dados no Cepel_Parte 2 (Avançada)(1-19).xlsx into table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_2_Avançada1_19
Successfully loaded Composição do Stack Tecnológico de Dados no Cepel_Parte 3 (Interface DevOps)(1-18).xlsx into table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps1_18

Tables in the database:
Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação1_21
Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_1_Geral1_19
Composição_do_Stack_Tecnológico_de_Da

In [3]:
conn.close()

In [4]:
example_table = tables[0][0]

cursor.execute(f"SELECT * FROM {example_table} LIMIT 1")
print([c[0] for c in cursor.description])

NameError: name 'tables' is not defined

In [ ]:
example_table = tables[0][0]

df = pd.read_sql_query(f"""
SELECT 
                       HA_QUANTO_TEMPO_VOCE_JA_TRABALHA_COM_DADOS_GOVERNANCA_ENGENHARIA_CIENCIA_OU_ANALISE_DE_DADOS, 
                       HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS,
                       NAME,
                       COUNT(*)
FROM {example_table}
GROUP BY NAME
""", conn)

df.head()

In [ ]:
# Example: Query the first few rows from the first available table (if any) and return as DataFrame
if tables:
    example_table = tables[0][0]  # Take the first table name from the list
    try:
        # Use pandas to directly read the SQL query result into a DataFrame
        df = pd.read_sql_query(f"SELECT * FROM {example_table} LIMIT 5", conn)
        print(f"\nFirst 5 rows from table {example_table} as a DataFrame:")
        display(df)  # Use display() for better formatting in Jupyter Notebook
    except Exception as e:
        print(f"Error querying table {example_table}: {str(e)}")
else:
    print("\nNo tables found in the database. Please check if the files were loaded correctly.")

In [6]:
# df = pd.read_sql_query(f"SELECT * FROM {example_table} LIMIT 5", conn)

In [25]:
example_table = tables[0][0]

df = pd.read_sql_query(f"""
SELECT
                       ID,
                       EMAIL,
                       NAME
                       EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO,
                       FAVOR_APONTAR_SUA_FAIXA_ETARIA,
                       FAVOR_APONTAR_O_SEU_TEMPO_DE_CASA_NO_CEPEL,
                       FAVOR_APONTAR_SEU_MAIOR_NIVEL_DE_FORMACAO_EDUCACIONAL,
                       HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS,
                       HA_QUANTO_TEMPO_VOCE_JA_TRABALHA_COM_DADOS_GOVERNANCA_ENGENHARIA_CIENCIA_OU_ANALISE_DE_DADOS,
                       COUNT(*)
FROM {example_table}

GROUP BY 
    HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS,
    EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO,
    ID,
    EMAIL,
    NAME
""", conn)

df

,ID,EMAIL,EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO,FAVOR_APONTAR_SUA_FAIXA_ETARIA,FAVOR_APONTAR_O_SEU_TEMPO_DE_CASA_NO_CEPEL,FAVOR_APONTAR_SEU_MAIOR_NIVEL_DE_FORMACAO_EDUCACIONAL,HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS,HA_QUANTO_TEMPO_VOCE_JA_TRABALHA_COM_DADOS_GOVERNANCA_ENGENHARIA_CIENCIA_OU_ANALISE_DE_DADOS,COUNT(*)
0,16,0360528@cepel.br,Claudio da Silva Pereira,35-44 anos,1-2 anos,Graduação,1-2 anos,Menos de 1 ano,1
1,25,0361500@cepel.br,Miguel Costa,18-24 anos,Menos de 1 ano,Ensino médio / profissionalizante,1-2 anos,Menos de 1 ano,1
2,24,0360704@cepel.br,Raul Procópio Mota,25-34 anos,1-2 anos,Graduação,3-5 anos,Menos de 1 ano,1
3,14,0360708@cepel.br,Wilson Albuquerque Ramos,25-34 anos,3-5 anos,Ensino médio / profissionalizante,3-5 anos,3-5 anos,1
4,23,0361970@cepel.br,Luis Sa,25-34 anos,1-2 anos,Graduação,3-5 anos,1-2 anos,1
5,7,0360841@cepel.br,Nério José Fernandes dos Santos,25-34 anos,3-5 anos,Graduação,3-5 anos,3-5 anos,1
6,9,0360099@cepel.br,Guilherme Prudente Da Silva,25-34 anos,1-2 anos,Mestrado / MBA,5-10 anos,3-5 anos,1
7,12,0360515@cepel.br,Ighor Souza dos Santos,25-34 anos,6-10 anos,Mestrado / MBA,5-10 anos,5-10 anos,1
8,11,0361934@cepel.br,Alberto Pavim,45-54 anos,Menos de 1 ano,Doutorado,5-10 anos,3-5 anos,1
9,13,0362504@cepel.br,Zhang Yuan,35-44 anos,Menos de 1 ano,Especialização,5-10 anos,5-10 anos,1


In [26]:
table_0 = tables[0][0]  # Take the first table name from the list
 # Use pandas to directly read the SQL query result into a DataFrame
df_0 = pd.read_sql_query(f"SELECT * FROM {table_0}", conn)

table_1 = tables[1][0]  # Take the first table name from the list
 # Use pandas to directly read the SQL query result into a DataFrame
df_1 = pd.read_sql_query(f"SELECT * FROM {table_1}", conn)

table_2 = tables[2][0]  # Take the first table name from the list
 # Use pandas to directly read the SQL query result into a DataFrame
df_2 = pd.read_sql_query(f"SELECT * FROM {table_2}", conn)

table_3 = tables[3][0]  # Take the first table name from the list
 # Use pandas to directly read the SQL query result into a DataFrame
df_3 = pd.read_sql_query(f"SELECT * FROM {table_3}", conn)

In [50]:
'''
# Join all 4 tables using Email as the common key
def join_all_four_tables(df_0, df_1, df_2, df_3):
    """
    Join all four DataFrames using Email as the common key
    """
    print("Starting to join all 4 tables...")
    print(f"df_0 shape: {df_0.shape}")
    print(f"df_1 shape: {df_1.shape}")
    print(f"df_2 shape: {df_2.shape}")
    print(f"df_3 shape: {df_3.shape}")
    
    # Step 1: Join df_0 with df_1
    joined_df = pd.merge(
        df_0,
        df_1,
        on='Email',
        how='left',
        suffixes=('', '_part1')
    )
    print(f"After joining df_0 + df_1: {joined_df.shape}")
    
    # Step 2: Join result with df_2
    joined_df = pd.merge(
        joined_df,
        df_2,
        on='Email',
        how='left',
        suffixes=('', '_part2')
    )
    print(f"After joining + df_2: {joined_df.shape}")
    
    # Step 3: Join result with df_3
    joined_df = pd.merge(
        joined_df,
        df_3,
        on='Email',
        how='left',
        suffixes=('', '_part3')
    )
    print(f"Final joined DataFrame shape: {joined_df.shape}")
    
    return joined_df

# Perform the complete join
joined_df = join_all_four_tables(df_0, df_1, df_2, df_3)
'''


# Step 1: Join df_0 with df_1
joined_df = pd.merge(
    df_0,
    df_1,
    on='ID',
    how='outer',
    suffixes=('', '_part1')
)
print(f"After joining df_0 + df_1: {joined_df.shape}")

# Step 2: Join result with df_2
joined_df = pd.merge(
    joined_df,
    df_2,
    on='ID',
    how='outer',
    suffixes=('', '_part2')
)
print(f"After joining + df_2: {joined_df.shape}")

# Step 3: Join result with df_3
joined_df = pd.merge(
    joined_df,
    df_3,
    on='ID',
    how='outer',
    suffixes=('', '_part3')
)

After joining df_0 + df_1: (21, 153)
After joining + df_2: (21, 268)


In [52]:
conn.close()

In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("cepel_data.db")

tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

print(tables)

                                                name
0  Composição_do_Stack_Tecnológico_de_Dados_no_Ce...
1  Composição_do_Stack_Tecnológico_de_Dados_no_Ce...
2  Composição_do_Stack_Tecnológico_de_Dados_no_Ce...
3  Composição_do_Stack_Tecnológico_de_Dados_no_Ce...
4                                          joined_df


In [ ]:
import sqlite3

conn = sqlite3.connect("cepel_data.db")

In [ ]:
#conn.close()

import sqlite3
conn = sqlite3.connect("cepel_data.db")

#conn.execute("DROP TABLE IF EXISTS joined_df")

joined_df.to_sql("joined_df", conn, if_exists="replace", index=False)

#pd.read_sql_query("SELECT * FROM joined_df", conn)

'''
Fluxo correto:

Pandas DataFrame
      ↓
to_sql()
      ↓
SQLite Table
      ↓
read_sql_query()

DatabaseError: Execution failed on sql 'DROP TABLE "joined_df"': database is locked
'''

In [ ]:
df_ = pd.read_sql_query(f"""SELECT * FROM joined_df""", conn)
df_.head()

In [14]:
#cols = [c for c in joined_df.columns if c != "ID"]

df_joined = pd.read_sql_query(f"""SELECT * FROM joined_df""", conn)
cols = [c for c in df_joined.columns if c not in ["ID", "EMAIL", "START_TIME", "COMPLETION_TIME", "NAME", "LAST_MODIFIED_TIME", "FAVOR_INSERIR_SEU_NOME_COMPLETO", "FAVOR_INSERIR_SEU_EMAIL_CEPELBR"]]
cols_sql = ", ".join(cols)
cols_sql

'EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO, FAVOR_APONTAR_SUA_FAIXA_ETARIA, FAVOR_APONTAR_O_SEU_TEMPO_DE_CASA_NO_CEPEL, FAVOR_APONTAR_SEU_MAIOR_NIVEL_DE_FORMACAO_EDUCACIONAL, HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS, HA_QUANTO_TEMPO_VOCE_JA_TRABALHA_COM_DADOS_GOVERNANCA_ENGENHARIA_CIENCIA_OU_ANALISE_DE_DADOS, ID_part1, START_TIME_part1, COMPLETION_TIME_part1, NAME_part1, LAST_MODIFIED_TIME_part1, ESTRUTURADOS_TABULARES_CSV_PLANILHAS_DATAFRAMES_SQL_DUMPS, SEMIESTRUTURADOS_SEM_ESQUEMA_RIGIDO_JSON_TOON_XML_PARQUET_ORC_AVRO, NAOESTRUTURADOS_SEM_FORMATO_DEFINIDO_TEXTOSDOCS_LOGS_AUDIOS_IMAGENS_VIDEOS_BINARIOS, QUAIS_DOS_SEGUINTES_TIPOS_DE_DADOS_LHE_SAO_PRIORITARIOS, ARQUIVOS_DE_DADOS_XLS_CSV_JSON_PARQUET_IMAGEM, SISTEMAS_CORPORATIVOS_ERP_CRM, APIS_E_SERVICOS_WEB, DADOS_DE_SENSORES_IOT, LOGS_DE_SERVIDORES_E_APLICACOES, DADOS_DE_REDES_SOCIAIS, DADOS_GEOESPACIAIS, BANCOS_DE_DADOS_SQL_E_NOSQL, OUTRA, SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRA_FONTE_DE_DADO_A

In [ ]:
df__new = pd.read_sql_query(f"""
SELECT {cols_sql},
    GROUP_CONCAT(DISTINCT FAVOR_APONTAR_SEU_MAIOR_NIVEL_DE_FORMACAO_EDUCACIONAL) AS FORMACAO_EDUCACIONA,
    COUNT(*) as total
FROM joined_df
GROUP BY 
    HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS,
    EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO
""", conn)

df__new

In [23]:
df__new = pd.read_sql_query(f"""
SELECT *,
    COUNT(*) as total
FROM joined_df
GROUP BY 
    HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS,
    EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO,
    NAME
""", conn)

# Versão alternativa para melhor compatibilidade com Excel no Brasil/Europa
df__new.to_csv(
    "analise_identificacao_e_geral_excel.csv",
    index=False,
    encoding='utf-8-sig',
    sep=';'  # Define o separador como ponto e vírgula
)

print("Versão para Excel salva com sucesso!")

Versão para Excel salva com sucesso!


In [ ]:
df__new

In [ ]:
#cols = [c for c in joined_df.columns if c != "ID"]
cols = [c for c in joined_df.columns if c not in ["ID", "EMAIL"]]
cols_sql = ", ".join(cols)
cols_sql

'START_TIME, COMPLETION_TIME, NAME, LAST_MODIFIED_TIME, FAVOR_INSERIR_SEU_NOME_COMPLETO, FAVOR_INSERIR_SEU_EMAIL_CEPELBR, EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO, FAVOR_APONTAR_SUA_FAIXA_ETARIA, FAVOR_APONTAR_O_SEU_TEMPO_DE_CASA_NO_CEPEL, FAVOR_APONTAR_SEU_MAIOR_NIVEL_DE_FORMACAO_EDUCACIONAL, HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS, HA_QUANTO_TEMPO_VOCE_JA_TRABALHA_COM_DADOS_GOVERNANCA_ENGENHARIA_CIENCIA_OU_ANALISE_DE_DADOS, ID_part1, START_TIME_part1, COMPLETION_TIME_part1, NAME_part1, LAST_MODIFIED_TIME_part1, ESTRUTURADOS_TABULARES_CSV_PLANILHAS_DATAFRAMES_SQL_DUMPS, SEMIESTRUTURADOS_SEM_ESQUEMA_RIGIDO_JSON_TOON_XML_PARQUET_ORC_AVRO, NAOESTRUTURADOS_SEM_FORMATO_DEFINIDO_TEXTOSDOCS_LOGS_AUDIOS_IMAGENS_VIDEOS_BINARIOS, QUAIS_DOS_SEGUINTES_TIPOS_DE_DADOS_LHE_SAO_PRIORITARIOS, ARQUIVOS_DE_DADOS_XLS_CSV_JSON_PARQUET_IMAGEM, SISTEMAS_CORPORATIVOS_ERP_CRM, APIS_E_SERVICOS_WEB, DADOS_DE_SENSORES_IOT, LOGS_DE_SERVIDORES_E_APLICACOES, DADOS_DE_REDES_SOCIAIS, DADOS_GEOESPA

In [45]:

df__new = pd.read_sql_query(f"""
SELECT 
    {cols_sql},
    COUNT(*) as total
FROM joined_df
GROUP BY 
    HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS,
    EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO
""", conn)

In [25]:
# Versão alternativa para melhor compatibilidade com Excel no Brasil/Europa
joined_df.to_csv(
    "analise_identificacao_e_geral_excel.csv",
    index=False,
    encoding='utf-8-sig',
    sep=';'  # Define o separador como ponto e vírgula
)

print("Versão para Excel salva com sucesso!")

Versão para Excel salva com sucesso!


In [20]:
#pd.set_option('display.max_columns', None)

In [25]:
import pandas as pd
import sqlite3
import os
import re
import unicodedata

def normalize_column(col):

    col = col.replace("\xa0", " ")
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("ascii")
    col = col.lower()
    col = re.sub(r"[^\w\s]", "", col)
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"_+", "_", col)

    return col.strip("_").upper()


directory = "files"

file_names = [
    "Composição do Stack Tecnológico de Dados no Cepel_Parte 0 (Identificação)(1-21).xlsx",
    "Composição do Stack Tecnológico de Dados no Cepel_Parte 1 (Geral)(1-19).xlsx",
    "Composição do Stack Tecnológico de Dados no Cepel_Parte 2 (Avançada)(1-19).xlsx",
    "Composição do Stack Tecnológico de Dados no Cepel_Parte 3 (Interface DevOps)(1-18).xlsx"
]

db_path = "cepel_data.db"
conn = sqlite3.connect(db_path)

print(f"Connected to SQLite database at {db_path}")

# carregar excel -> sqlite
for file_name in file_names:

    file_path = os.path.join(directory, file_name)

    df = pd.read_excel(file_path)
    df.columns = [normalize_column(c) for c in df.columns]

    table_name = file_name.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_").split(".")[0]

    df.to_sql(table_name, conn, if_exists="replace", index=False)

    print(f"Loaded {file_name} -> table {table_name}")


# pegar todas as tabelas
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("\nTables in database:")
for table in tables:
    print(table[0])


# salvar todas as tabelas em CSV
for table in tables:

    table_name = table[0]

    df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)

    file_output = f"{table_name}.csv"

    df.to_csv(
        file_output,
        index=False,
        encoding="utf-8-sig",
        sep=";"
    )

    print(f"Tabela {table_name} salva em {file_output}")


print("\nTodas as tabelas foram exportadas com sucesso!")

Connected to SQLite database at cepel_data.db
Loaded Composição do Stack Tecnológico de Dados no Cepel_Parte 0 (Identificação)(1-21).xlsx -> table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação1_21
Loaded Composição do Stack Tecnológico de Dados no Cepel_Parte 1 (Geral)(1-19).xlsx -> table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_1_Geral1_19
Loaded Composição do Stack Tecnológico de Dados no Cepel_Parte 2 (Avançada)(1-19).xlsx -> table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_2_Avançada1_19
Loaded Composição do Stack Tecnológico de Dados no Cepel_Parte 3 (Interface DevOps)(1-18).xlsx -> table Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps1_18

Tables in database:
joined_df
Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação1_21
Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_1_Geral1_19
Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_2_Avançada1_19
Composição_do_Stack_

In [29]:
df__new = pd.read_sql_query(f"""
SELECT SPHINX,
    COUNT(*) as total
FROM joined_df
GROUP BY 
    HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS,
    EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO,
    NAME,
    SPHINX
""", conn)
